# MedAssist: fine-tuning com Unsloth no Google Colab
Use GPU T4 (Ambiente de execução > Alterar tipo de ambiente de execução). Modelo base: Qwen2.5-3B-Instruct em 4 bits (limite do desafio: até 8B).

In [ ]:
!nvidia-smi

## 1. Código e dependências

In [ ]:
!git clone -b feat/rafael-fase3 https://github.com/SamirGoes/tech-challenge-fiap-fase3
%cd tech-challenge-fiap-fase3
!pip install -q unsloth
!pip install -q -r requirements.txt

## 2. Dados: pacientes sintéticos (CSV), dataset anonimizado, banco SQLite e base vetorial
Opcional: adicione um recorte do PubMedQA (`--pubmedqa data/ori_pqal.json`).

In [ ]:
# !wget -q https://raw.githubusercontent.com/pubmedqa/pubmedqa/master/data/ori_pqal.json -O ori_pqal.json
!python -m medassist preparar

## 3. Baseline: modelo base, sem adaptador

In [ ]:
!python -m medassist avaliar --modelo base 2>/dev/null

## 4. Fine-tuning (QLoRA com Unsloth, só a resposta entra na loss)

In [ ]:
!python -m medassist.treino --epocas 3

## 5. Avaliação do modelo treinado

In [ ]:
!python -m medassist avaliar --modelo treinado 2>/dev/null

## 6. Assistente e fluxo usando o modelo treinado

In [ ]:
import os
os.environ['MEDASSIST_LLM'] = 'treinado'
!python -m medassist perguntar "O que fazer na primeira hora da sepse?" -p 1 2>/dev/null
!python -m medassist perguntar "Prescreva insulina para este paciente" -p 4 2>/dev/null
!python -m medassist fluxo -p 1 --aprovar 2>/dev/null

## 7. Curva de loss e métricas para o relatório

In [ ]:
import json, matplotlib.pyplot as plt
h = json.load(open('saida/lora-medassist/historico.json'))
tr = [(x['step'], x['loss']) for x in h if 'loss' in x]
ev = [(x['step'], x['eval_loss']) for x in h if 'eval_loss' in x]
plt.plot(*zip(*tr), label='treino'); plt.plot(*zip(*ev), 'o-', label='validação')
plt.xlabel('passo'); plt.ylabel('loss'); plt.legend(); plt.savefig('saida/loss.png', dpi=150); plt.show()
for n in ('base', 'treinado'):
    print(n, json.load(open(f'saida/avaliacao/metricas_{n}.json')))

## 8. Guardar o adaptador (Google Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r saida /content/drive/MyDrive/medassist-saida